In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [3]:
train_df = pd.read_csv("ielts_train_df.csv")
eval_df  = pd.read_csv("ielts_val_df.csv")
test_df  = pd.read_csv("ielts_test_df.csv")


In [4]:
kaggle_raw = pd.read_csv("ielts_writing_dataset.csv")

print("Raw shape:", kaggle_raw.shape)
print(kaggle_raw["Task_Type"].value_counts(dropna=False).sort_index())

# Keep only Task 2
kaggle_t2 = kaggle_raw[kaggle_raw["Task_Type"] == 2].copy().reset_index(drop=True)

Raw shape: (2162, 14)
Task_Type
1     822
2    1340
Name: count, dtype: int64


In [5]:
def safe_word_count(text):
    if pd.isna(text):
        return np.nan
    return len(str(text).split())

def build_target_text(prompt, essay):
    prompt = "" if pd.isna(prompt) else str(prompt).strip()
    essay  = "" if pd.isna(essay) else str(essay).strip()
    return f"[PROMPT]\n{prompt}\n\n[ESSAY]\n{essay}"

def build_evaluation(row):
    parts = []
    if pd.notna(row.get("justification_TR")):
        parts.append(f"Task Response:\n{row['justification_TR']}")
    if pd.notna(row.get("justification_CC")):
        parts.append(f"Coherence and Cohesion:\n{row['justification_CC']}")
    if pd.notna(row.get("justification_LR")):
        parts.append(f"Lexical Resource:\n{row['justification_LR']}")
    if pd.notna(row.get("justification_GRA")):
        parts.append(f"Grammar Range and Accuracy:\n{row['justification_GRA']}")
    if pd.notna(row.get("overall_verification")):
        parts.append(f"Overall Verification:\n{row['overall_verification']}")
    return "\n\n".join(parts).strip()

In [6]:
kaggle_df = pd.DataFrame({
    "prompt": kaggle_t2["Question"].astype(str).str.strip(),
    "essay": kaggle_t2["Essay"].astype(str).str.strip(),
    "evaluation": kaggle_t2.apply(build_evaluation, axis=1),
    "essay_len": kaggle_t2["Essay"].apply(safe_word_count),
    "TR": kaggle_t2["Task_Response"].astype(float),
    "CC": kaggle_t2["Coherence_Cohesion"].astype(float),
    "LR": kaggle_t2["Lexical_Resource"].astype(float),
    "GRA": kaggle_t2["Range_Accuracy"].astype(float),
    "n_found": 4,
    "parse_quality": 1.0,
    "overall_raw": kaggle_t2["Overall"].astype(float),
    "band": kaggle_t2["Overall"].astype(float),
    "prompt_relevance": np.nan,
    "lexical_diversity": np.nan,
    "readability_score": np.nan,
})

kaggle_df["target_text"] = [
    build_target_text(p, e) for p, e in zip(kaggle_df["prompt"], kaggle_df["essay"])
]
kaggle_df["source"] = "kaggle_v2"

In [7]:
for df in [train_df, eval_df, test_df]:
    if "source" not in df.columns:
        df["source"] = "hf"

required_cols = [
    "prompt", "essay", "evaluation", "essay_len",
    "TR", "CC", "LR", "GRA",
    "n_found", "parse_quality",
    "overall_raw", "band",
    "prompt_relevance", "lexical_diversity", "readability_score",
    "target_text", "source"
]

for name, df in [("train_df", train_df), ("eval_df", eval_df), ("test_df", test_df)]:
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"{name} missing columns: {missing}")

train_df = train_df[required_cols].copy()
eval_df  = eval_df[required_cols].copy()
test_df  = test_df[required_cols].copy()
kaggle_df = kaggle_df[required_cols].copy()

In [8]:
before = len(kaggle_df)
kaggle_df = kaggle_df.drop_duplicates(subset=["prompt", "essay"]).reset_index(drop=True)
after = len(kaggle_df)
print(f"Kaggle Task2 dedup inside: {before} -> {after}")

# =========================
# 7) Remove any overlap with existing HF splits
# =========================
all_existing = pd.concat([
    train_df[["prompt", "essay"]],
    eval_df[["prompt", "essay"]],
    test_df[["prompt", "essay"]],
], axis=0).drop_duplicates()

kaggle_df = kaggle_df.merge(
    all_existing.assign(_exists=1),
    on=["prompt", "essay"],
    how="left"
)
kaggle_df = kaggle_df[kaggle_df["_exists"].isna()].drop(columns=["_exists"]).reset_index(drop=True)

print("Kaggle Task2 after removing overlap with HF:", kaggle_df.shape)


Kaggle Task2 dedup inside: 1340 -> 1304
Kaggle Task2 after removing overlap with HF: (1304, 17)


In [9]:
kaggle_df["band_strat"] = kaggle_df["band"].astype(str)

try:
    kaggle_train_add, kaggle_eval_add = train_test_split(
        kaggle_df,
        test_size=0.15,
        random_state=42,
        stratify=kaggle_df["band_strat"]
    )
except:
    kaggle_train_add, kaggle_eval_add = train_test_split(
        kaggle_df,
        test_size=0.15,
        random_state=42
    )

kaggle_train_add = kaggle_train_add.drop(columns=["band_strat"], errors="ignore")
kaggle_eval_add  = kaggle_eval_add.drop(columns=["band_strat"], errors="ignore")

In [10]:
train_aug = pd.concat([train_df, kaggle_train_add], ignore_index=True)
eval_aug  = pd.concat([eval_df, kaggle_eval_add], ignore_index=True)

train_aug = train_aug.drop_duplicates(subset=["prompt", "essay"]).reset_index(drop=True)
eval_aug  = eval_aug.drop_duplicates(subset=["prompt", "essay"]).reset_index(drop=True)

In [11]:
train_aug.to_csv("ielts_train_aug_df.csv", index=False, encoding="utf-8-sig")
eval_aug.to_csv("ielts_evals_aug_df.csv", index=False, encoding="utf-8-sig")
test_df.to_csv("ielts_test_locked_df.csv", index=False, encoding="utf-8-sig")
kaggle_df.to_csv("ielts_kaggle_task2_mapped.csv", index=False, encoding="utf-8-sig")

print("\nSaved:")
print("- ielts_train_aug_df.csv")
print("- ielts_evals_aug_df.csv")
print("- ielts_test_locked_df.csv")
print("- ielts_kaggle_task2_mapped.csv")

print("\nShapes:")
print("train original:", train_df.shape)
print("eval original :", eval_df.shape)
print("test original :", test_df.shape)
print("train aug     :", train_aug.shape)
print("eval aug      :", eval_aug.shape)
print("test locked   :", test_df.shape)

print("\nSource counts in train_aug:")
print(train_aug["source"].value_counts(dropna=False))

print("\nSource counts in eval_aug:")
print(eval_aug["source"].value_counts(dropna=False))


Saved:
- ielts_train_aug_df.csv
- ielts_evals_aug_df.csv
- ielts_test_locked_df.csv
- ielts_kaggle_task2_mapped.csv

Shapes:
train original: (6412, 17)
eval original : (802, 17)
test original : (802, 17)
train aug     : (7520, 17)
eval aug      : (998, 17)
test locked   : (802, 17)

Source counts in train_aug:
source
hf           6412
kaggle_v2    1108
Name: count, dtype: int64

Source counts in eval_aug:
source
hf           802
kaggle_v2    196
Name: count, dtype: int64
